In [4]:
%pip install pandas
%pip install sqlalchemy
%pip install dbt-postgres

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Pipeline de Extração de Dados (Data Extraction)

In [6]:
# --- CONFIGURAR A CONEXÃO --- (Já está configurado do mesmo jeito que é descrito em "setup_banco.md", "4. Criar o banco de dados pelo pgAdmin")
usuario_db = 'postgres'
senha_db = 'postgres'
host_db = 'localhost'
porta_db = '5432'
nome_banco_db = 'procons_sindec'

DATABASE_URL = f'postgresql://{usuario_db}:{senha_db}@{host_db}:{porta_db}/{nome_banco_db}'
engine = create_engine(DATABASE_URL)

print("Conexão criada.")

# --- DICIONÁRIO DOS ARQUIVOS BRUTOS ---
arquivos_brutos = {
    '2009': '../dados_brutos/CNRF_2009.csv',
    '2010': '../dados_brutos/CNRF_2010.csv',
    '2011': '../dados_brutos/CNRF_2011.csv',
    '2012': '../dados_brutos/CNRF_2012.csv',
    '2013': '../dados_brutos/CNRF_2013.csv',
    '2014': '../dados_brutos/CNRF_2014.csv',
    '2015': '../dados_brutos/CNRF_2015.csv',
    '2016': '../dados_brutos/CNRF_2016.csv',
    '2017': '../dados_brutos/CNRF_2017.csv',
    '2018': '../dados_brutos/CNRF_2018.csv',
    '2019': '../dados_brutos/CNRF_2019.csv',
    '2020': '../dados_brutos/CNRF_2020.csv',
    '2021': '../dados_brutos/CNRF_2021.csv',
    '2022': '../dados_brutos/CNRF_2022.csv',
    '2023': '../dados_brutos/CNRF_2023.csv',
    '2024': '../dados_brutos/CNRF_2024.csv',
    '2023': '../dados_brutos/CNRF_2023.csv',
    '2024': '../dados_brutos/CNRF_2024.xlsx'
}

Conexão criada.


In [7]:
# Esse código demooooooora............................................................. (coisa de 3 minutos)
print("Iniciando carga dos dados BRUTOS...")

# --- EXTRAÇÃO e CARGA ---
colunas_padrao = [
    'anocalendario', 'dataarquivamento', 'dataabertura', 'codigoregiao', 'regiao',
    'uf', 'strrazaosocial', 'strnomefantasia', 'tipo', 'numerocnpj', 'radicalcnpj',
    'razaosocialrfb', 'nomefantasiarfb', 'cnaeprincipal', 'desccnaeprincipal',
    'atendida', 'codigoassunto', 'descricaoassunto', 'codigoproblema',
    'descricaoproblema', 'sexoconsumidor', 'faixaetariaconsumidor', 'cepconsumidor'
]


for ano, caminho_arquivo in arquivos_brutos.items():
    # Criando uma tabela crua para cada ano no PostgreSQL
    nome_tabela = f"raw_procons_{ano}"
    
    try:
        print(f"[{ano}] Extraindo: {caminho_arquivo}")
        
        # --- EXTRAÇÃO ---
        if caminho_arquivo.endswith('.csv'):
            if ano == '2017':
                # Caso especial em que o arquivo não apresenta header
                df_raw = pd.read_csv(caminho_arquivo, encoding='latin1', sep=';', header=None, names=colunas_padrao, dtype=str, on_bad_lines='skip')
            else:
                # Caso comum em que o arquivo apresenta header
                df_raw = pd.read_csv(caminho_arquivo, encoding='latin1', sep=';', dtype=str, on_bad_lines='skip')
                df_raw.columns = colunas_padrao
                
        elif caminho_arquivo.endswith('.xlsx'):
            df_raw = pd.read_excel(caminho_arquivo, dtype=str)
            df_raw.columns = colunas_padrao
                    
        # --- CARGA ---
        print(f"[{ano}] Carregando {len(df_raw)} registros na tabela '{nome_tabela}'...")
        
        df_raw.to_sql(nome_tabela, con=engine, if_exists='replace', index=False)
        
        print(f"[{ano}] Sucesso!\n")
        

    except Exception as e:
        print(f"[{ano}] Erro Crítico: {e}\n")

print("Processo EL Finalizado. O banco de dados está populado e pronto para o dbt")

Iniciando carga dos dados BRUTOS...
[2009] Extraindo: ../dados_brutos/CNRF_2009.csv
[2009] Carregando 104869 registros na tabela 'raw_procons_2009'...
[2009] Erro Crítico: (psycopg2.errors.DependentObjectsStillExist) cannot drop table raw_procons_2009 because other objects depend on it
DETAIL:  view dados_brutos.stg_reclamacoes_unificados depends on table raw_procons_2009
view dados_brutos.int_reclamacoes_preparadas depends on view dados_brutos.stg_reclamacoes_unificados
HINT:  Use DROP ... CASCADE to drop the dependent objects too.

[SQL: 
DROP TABLE raw_procons_2009]
(Background on this error at: https://sqlalche.me/e/20/2j85)

[2010] Extraindo: ../dados_brutos/CNRF_2010.csv
[2010] Carregando 122664 registros na tabela 'raw_procons_2010'...
[2010] Erro Crítico: (psycopg2.errors.DependentObjectsStillExist) cannot drop table raw_procons_2010 because other objects depend on it
DETAIL:  view dados_brutos.stg_reclamacoes_unificados depends on table raw_procons_2010
view dados_brutos.int_r